## Overview

1. Clustering               pair shortlisting, Dinov2
2. Local matching           SuperPoint + SuperGlue, or MASt3R (mby later lightglue instead of superglue)
3. Geometric verification   RANSAC 
4. Reconstruction           COLMAP/pycolmap structure-from-motion (SfM) → camera poses
                            each separate reconstruction = one scene cluster; unregistered = outlier


### First simple pipeline

DINOv2 embedding per image -> clustering by HDBSCAN(cosine) -> Create pairs within each cluster -> Keypoint extraction per image via SuperPoint -> Match Keypointlist per pair with SuperGlue -> 

## Pair shortlisting

Pair sortlisting via embedding using Dinov2
https://github.com/facebookresearch/dinov2

embed Images and group them into sets of similar images,
return a list of candidate pairs to match

In [18]:
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from pathlib import Path
import sys
import cv2
sys.path.append('SuperGluePretrainedNetwork')
from models.matching import Matching
from models.superpoint import SuperPoint
from models.superglue import SuperGlue
from models.utils import frame2tensor

## cd IMC2025-ML-Project
## git clone https://github.com/magicleap/SuperGluePretrainedNetwork.git


dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

Using cache found in /Users/lucbaumeler/.cache/torch/hub/facebookresearch_dinov2_main


In [19]:
dinov2.eval().to(device)

transform = T.Compose([T.Resize((448, 448)), T.ToTensor(), T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])



#### Example embeddings with cosine similarity

In [20]:
x1 = Image.open('image-matching-challenge-2025/test/stairs/stairs_split_1_1710453576271.png').convert('RGB')
x2 = Image.open('image-matching-challenge-2025/test/stairs/stairs_split_1_1710453601885.png').convert('RGB')
x3 = Image.open('image-matching-challenge-2025/test/ETs/another_et_another_et001.png').convert('RGB')

x1 = transform(x1).unsqueeze(0).to(device)
x2 = transform(x2).unsqueeze(0).to(device)
x3 = transform(x3).unsqueeze(0).to(device)



with torch.no_grad():
    y1 = dinov2(x1)
    y2 = dinov2(x2)
    y3 = dinov2(x3)
    
sim = F.cosine_similarity(y1, y2)
sim2 = F.cosine_similarity(y1, y3)
print(sim.item())
print(sim2.item())


0.8893311023712158
0.02078670635819435


**Create list of image embeddings and per folder**

In [21]:
ds = ImageFolder('image-matching-challenge-2025/train', transform)
loader = DataLoader(dataset=ds, batch_size=64, shuffle=False)

embs = []
labls = []

with torch.no_grad():
    for imgs, labels in loader:
        embs.append(dinov2(imgs.to(device)))
        labls.append(labels.to(device))

In [22]:
print(embs[0])
print(labls[0])

tensor([[ 0.6501, -3.5805, -1.6924,  ...,  1.5323, -0.8058, -1.3643],
        [-0.9997, -3.7093, -0.3785,  ...,  0.5888,  0.0951, -1.5165],
        [-0.8266, -3.5939, -1.2035,  ...,  1.7604,  1.4490, -1.6209],
        ...,
        [ 2.5760, -0.2871, -1.6254,  ...,  0.5779,  2.3726,  3.6004],
        [ 1.4662, -0.4712, -2.8981,  ...,  1.2748,  3.1438,  3.2336],
        [ 1.3754, -0.6801, -1.5857,  ...,  2.2001,  3.8991,  2.8979]])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


In [23]:
E = torch.cat(embs)
L = torch.cat(labls)
emb = {name: E[L == idx] for name, idx in ds.class_to_idx.items()}
paths = {name: [p for p, l in ds.samples if l == idx] for name, idx in ds.class_to_idx.items()}

for name, X in emb.items():
    print(name, X.shape, len(paths[name]))

ETs torch.Size([22, 384]) 22
amy_gardens torch.Size([200, 384]) 200
fbk_vineyard torch.Size([163, 384]) 163
imc2023_haiper torch.Size([54, 384]) 54
imc2023_heritage torch.Size([209, 384]) 209
imc2023_theather_imc2024_church torch.Size([76, 384]) 76
imc2024_dioscuri_baalshamin torch.Size([138, 384]) 138
imc2024_lizard_pond torch.Size([214, 384]) 214
pt_brandenburg_british_buckingham torch.Size([225, 384]) 225
pt_piazzasanmarco_grandplace torch.Size([168, 384]) 168
pt_sacrecoeur_trevi_tajmahal torch.Size([225, 384]) 225
pt_stpeters_stpauls torch.Size([200, 384]) 200
stairs torch.Size([51, 384]) 51


In [24]:
from sklearn.cluster import HDBSCAN
X = F.normalize(emb['ETs'], dim=1).numpy()
labels = HDBSCAN(min_cluster_size=3, metric='cosine').fit_predict(X)

print(labels)

[ 0  0  0  0  0  0  0  0  0  0  2  2  2  2  2  1  1  1 -1 -1 -1 -1]


/opt/anaconda3/envs/imc2025/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


**Cluester the images using HDBSCAN into images of the same object and outliers per dataset**

In [25]:
clusters = {}
for name, X in emb.items():
    X = F.normalize(X, dim=1).numpy()
    labels = HDBSCAN(min_cluster_size=3, metric='euclidean').fit_predict(X)

    groups = {}
    for file, lab in zip(paths[name], labels):
        scene = 'outliers' if lab == -1 else f'cluster{lab}'
        groups.setdefault(scene, []).append(Path(file).name)

    clusters[name] = groups

for name, groups in clusters.items():
    print(name, {scene: files for scene, files in groups.items()})



ETs {'cluster0': ['another_et_another_et001.png', 'another_et_another_et002.png', 'another_et_another_et003.png', 'another_et_another_et004.png', 'another_et_another_et005.png', 'another_et_another_et006.png', 'another_et_another_et007.png', 'another_et_another_et008.png', 'another_et_another_et009.png', 'another_et_another_et010.png'], 'cluster1': ['et_et000.png', 'et_et001.png', 'et_et002.png', 'et_et003.png', 'et_et004.png', 'et_et005.png', 'et_et006.png', 'et_et007.png', 'et_et008.png'], 'outliers': ['outliers_out_et001.png', 'outliers_out_et002.png', 'outliers_out_et003.png']}
amy_gardens {'cluster1': ['peach_0000.png', 'peach_0001.png', 'peach_0002.png', 'peach_0006.png', 'peach_0007.png', 'peach_0011.png', 'peach_0012.png', 'peach_0013.png', 'peach_0014.png', 'peach_0016.png', 'peach_0017.png', 'peach_0018.png', 'peach_0020.png', 'peach_0021.png', 'peach_0023.png', 'peach_0024.png', 'peach_0025.png', 'peach_0026.png', 'peach_0028.png', 'peach_0029.png', 'peach_0030.png', 'peach_

/opt/anaconda3/envs/imc2025/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/opt/anaconda3/envs/imc2025/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/opt/anaconda3/envs/imc2025/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/opt/anaconda3/envs/imc2025/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/opt/ana

In [26]:
from itertools import combinations

pairs = {}

for name, groups in clusters.items():
    pairs[name] = {}
    for scene, files in groups.items():
        if scene == 'outliers':
            continue
        else:
            pairs[name][scene] = list(combinations(sorted(files), 2))


### Keypoint extraction + Matching

https://github.com/magicleap/SuperGluePretrainedNetwork

Mby use superpoint once per image, save keypoints and then run superglue per pair for efficiency if needed.
mby change outdoor/indoor per scene

In [ ]:
matching = Matching({'superpoint': {'max_keypoints': 512},
                     'superglue':   {'weights': 'outdoor', 'sinkhorn_iterations': 20}}).eval().to(device)

keypoints = []


with torch.no_grad():
    for name in pairs:
            for scene, pair_list in pairs[name].items():
                for a, b in pair_list:
                    A = cv2.imread(f'image-matching-challenge-2025/train/{name}/{a}', cv2.IMREAD_GRAYSCALE)
                    B = cv2.imread(f'image-matching-challenge-2025/train/{name}/{b}', cv2.IMREAD_GRAYSCALE)
                    pred = matching({'image0': frame2tensor(A, device), 'image1': frame2tensor(B, device)})

                    keypoints.append({'pair': (a, b), 'keypoints0': pred['keypoints0'][0].cpu().numpy(),
                                      'keypoints1': pred['keypoints1'][0].cpu().numpy(),
                                      'matches0': pred['matches0'].cpu().numpy(),
                                      'scores0': pred['matching_scores0'][0].cpu().numpy()})


Loaded SuperPoint model
Loaded SuperGlue model ("outdoor" weights)
